# Notebook 4 — Cross-Dataset Generalization
**Project:** Chest X-ray Pneumonia Detection — Generalization Study

This notebook answers the core research question:
> **Do K-Means features learned on Kaggle chest X-rays generalize to a completely different hospital dataset (NIH)?**

We evaluate the best model (tuned XGBoost) on the NIH dataset and compare
performance against the Kaggle test set to measure the generalization gap.

In [ ]:
!pip install scikit-learn xgboost numpy matplotlib seaborn --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, roc_curve)
from xgboost import XGBClassifier

SEED = 42
print('Libraries loaded.')

## Step 1 — Load features, best model, and scaler

In [ ]:
# Load features
X_train_feat = np.load('preprocessed/X_train_feat.npy')
X_test_feat  = np.load('preprocessed/X_test_feat.npy')
X_nih_feat   = np.load('preprocessed/X_nih_feat.npy')
y_train      = np.load('preprocessed/y_train.npy')
y_test       = np.load('preprocessed/y_test.npy')
y_nih        = np.load('preprocessed/y_nih.npy')

# Load scaler and best model
with open('models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('models/best_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

# Scale features using the SAME scaler trained on Kaggle data
X_test_s = scaler.transform(X_test_feat)
X_nih_s  = scaler.transform(X_nih_feat)  # never seen during training

print('Everything loaded.')
print(f'NIH test set: {X_nih_s.shape[0]} images  (Normal: {(y_nih==0).sum()}, Pneumonia: {(y_nih==1).sum()})')

## Step 2 — Evaluate all models on NIH (generalization test)

In [ ]:
def get_metrics(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1] if hasattr(model, 'predict_proba') else None
    return {
        'accuracy':  accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall':    recall_score(y, y_pred, zero_division=0),
        'f1':        f1_score(y, y_pred, zero_division=0),
        'roc_auc':   roc_auc_score(y, y_prob) if y_prob is not None else 0.0,
        'y_pred':    y_pred,
        'y_prob':    y_prob
    }

# Evaluate best model on both test sets
kaggle_metrics = get_metrics(best_model, X_test_s, y_test)
nih_metrics    = get_metrics(best_model, X_nih_s,  y_nih)

print('=== Best Model (XGBoost tuned) — Generalization Results ===')
print(f'  Kaggle test  :  Acc={kaggle_metrics["accuracy"]:.4f}  F1={kaggle_metrics["f1"]:.4f}  AUC={kaggle_metrics["roc_auc"]:.4f}')
print(f'  NIH test     :  Acc={nih_metrics["accuracy"]:.4f}  F1={nih_metrics["f1"]:.4f}  AUC={nih_metrics["roc_auc"]:.4f}')
print(f'  Acc gap      :  {kaggle_metrics["accuracy"] - nih_metrics["accuracy"]:+.4f}')
print(f'  AUC gap      :  {kaggle_metrics["roc_auc"]  - nih_metrics["roc_auc"]:+.4f}')

## Step 3 — Compare all models: Kaggle vs NIH performance

In [ ]:
from sklearn.preprocessing import StandardScaler as SS

# Re-train all models quickly (using saved scaler)
X_train_s = scaler.transform(X_train_feat)

models = {
    'Logistic Regression': LogisticRegression(random_state=SEED, max_iter=1000, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=SEED, class_weight='balanced'),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=SEED, class_weight='balanced'),
    'XGBoost (tuned)':     best_model
}

gen_results = []
print(f'{"Model":<25} {"Kaggle Acc":>10} {"NIH Acc":>10} {"Acc Gap":>10} {"Kaggle AUC":>12} {"NIH AUC":>10} {"AUC Gap":>10}')
print('-' * 90)

for name, model in models.items():
    if name != 'XGBoost (tuned)':
        model.fit(X_train_s, y_train)
    
    k_met = get_metrics(model, X_test_s, y_test)
    n_met = get_metrics(model, X_nih_s,  y_nih)
    acc_gap = k_met['accuracy'] - n_met['accuracy']
    auc_gap = k_met['roc_auc']  - n_met['roc_auc']
    
    print(f'{name:<25} {k_met["accuracy"]:>10.4f} {n_met["accuracy"]:>10.4f} {acc_gap:>+10.4f} {k_met["roc_auc"]:>12.4f} {n_met["roc_auc"]:>10.4f} {auc_gap:>+10.4f}')
    gen_results.append({'model': name, 'kaggle_acc': k_met['accuracy'], 'nih_acc': n_met['accuracy'],
                        'acc_gap': acc_gap, 'kaggle_auc': k_met['roc_auc'],
                        'nih_auc': n_met['roc_auc'], 'auc_gap': auc_gap,
                        'kaggle_metrics': k_met, 'nih_metrics': n_met})

## Step 4 — Visualize generalization gap

In [ ]:
model_names = [r['model'] for r in gen_results]
kaggle_aucs = [r['kaggle_auc'] for r in gen_results]
nih_aucs    = [r['nih_auc']    for r in gen_results]
auc_gaps    = [r['auc_gap']    for r in gen_results]

x = np.arange(len(model_names))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Cross-Dataset Generalization Analysis', fontsize=13)

# Panel 1: Kaggle AUC
axes[0].bar(x, kaggle_aucs, color='#2196F3', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('ROC-AUC'); axes[0].set_title('Kaggle Test AUC')
axes[0].set_ylim(0.5, 1.0); axes[0].grid(True, alpha=0.3, axis='y')

# Panel 2: Kaggle vs NIH side by side
bars1 = axes[1].bar(x - width/2, kaggle_aucs, width, label='Kaggle', color='#2196F3', alpha=0.85)
bars2 = axes[1].bar(x + width/2, nih_aucs,    width, label='NIH',    color='#F44336', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
axes[1].set_ylabel('ROC-AUC'); axes[1].set_title('Kaggle vs NIH AUC')
axes[1].legend(); axes[1].set_ylim(0.5, 1.0); axes[1].grid(True, alpha=0.3, axis='y')

# Panel 3: Generalization gap
colors_gap = ['#4CAF50' if g < 0.05 else '#FF9800' if g < 0.1 else '#F44336' for g in auc_gaps]
axes[2].bar(x, auc_gaps, color=colors_gap, alpha=0.85)
axes[2].set_xticks(x); axes[2].set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
axes[2].set_ylabel('AUC Gap (Kaggle - NIH)'); axes[2].set_title('Generalization Gap (AUC)')
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].axhline(y=0.05, color='orange', linestyle='--', alpha=0.7, label='0.05 threshold')
axes[2].legend()

plt.tight_layout()
plt.savefig('generalization_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as generalization_gap.png')

## Step 5 — Confusion matrix on NIH dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('XGBoost (tuned) — Kaggle vs NIH Confusion Matrices', fontsize=12)

for ax, metrics, title in zip(axes,
                               [kaggle_metrics, nih_metrics],
                               ['Kaggle Test Set', 'NIH Dataset (Generalization)']):
    cm = confusion_matrix(y_test if title.startswith('Kaggle') else y_nih, metrics['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal','Pneumonia'],
                yticklabels=['Normal','Pneumonia'])
    ax.set_title(title); ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('nih_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as nih_confusion_matrices.png')
print('Notebook 4 complete. Open notebook 5 (visualizations) next.')